In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator,load_img, img_to_array
import numpy as np
import os
from tensorflow.keras.utils import Sequence
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint

In [5]:

# Modell: U-Net für Segmentierung
def unet(input_size=(256, 256, 3)):
    inputs = layers.Input(input_size)

    # Encoder (Downsampling)
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c4)
    p4 = layers.MaxPooling2D((2, 2))(c4)

    # Bottleneck
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(p4)
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(c5)

    # Decoder (Upsampling)
    u6 = layers.UpSampling2D((2, 2))(c5)
    u6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(u6)
    u6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(u6)
    u6 = layers.concatenate([u6, c4], axis=3)

    u7 = layers.UpSampling2D((2, 2))(u6)
    u7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(u7)
    u7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(u7)
    u7 = layers.concatenate([u7, c3], axis=3)

    u8 = layers.UpSampling2D((2, 2))(u7)
    u8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u8)
    u8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u8)
    u8 = layers.concatenate([u8, c2], axis=3)

    u9 = layers.UpSampling2D((2, 2))(u8)
    u9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u9)
    u9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u9)
    u9 = layers.concatenate([u9, c1], axis=3)

    outputs = layers.Conv2D(4, (1, 1), activation='softmax')(u9)  # 4 Klassen

    model = models.Model(inputs, outputs)

    return model


In [ ]:

class ImageMaskGenerator(Sequence):
    def __init__(self, image_dir, mask_dir, batch_size=32, target_size=(512, 512), shuffle=True, num_classes=4):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.batch_size = batch_size
        self.target_size = target_size
        self.shuffle = shuffle
        self.num_classes = num_classes
        
        # List image and mask filenames
        self.image_filenames = sorted(os.listdir(image_dir))
        self.mask_filenames = sorted(os.listdir(mask_dir))

        self.indexes = np.arange(len(self.image_filenames))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.image_filenames) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        # Get batch of image and mask pairs
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        images = []
        masks = []

        for idx in batch_indexes:
            image_path = os.path.join(self.image_dir, self.image_filenames[idx])
            mask_path = os.path.join(self.mask_dir, self.mask_filenames[idx])

            # Load image and mask
            image = load_img(image_path, target_size=self.target_size)
            image = img_to_array(image) / 255.0  # Normalize image

            mask = load_img(mask_path, target_size=self.target_size, color_mode="grayscale")
            mask = img_to_array(mask)  # Masks are grayscale, so we can keep them in 1 channel
            mask = np.squeeze(mask)  # Remove the last channel if it's (height, width, 1)

            # One-hot encode the mask
            mask_one_hot = np.zeros((self.target_size[0], self.target_size[1], self.num_classes), dtype=np.float32)
            for c in range(self.num_classes):
                mask_one_hot[..., c] = (mask == c).astype(np.float32)  # Set 1 for pixels of class `c`

            images.append(image)
            masks.append(mask_one_hot)

        return np.array(images), np.array(masks)

# Example usage:
image_dir = './images/satellites'  # Path to the folder with satellite images
mask_dir = './images/masks'  # Path to the folder with mask images

train_generator = ImageMaskGenerator(image_dir=image_dir, mask_dir=mask_dir, batch_size=8)

# Define the U-Net model (or any other model you want)
model = unet(input_size=(512, 512, 3))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


# Define the model checkpoint callback to save the best model based on validation accuracy
checkpoint_callback = ModelCheckpoint(
    'best_model.h5',            # Path where the model will be saved
    monitor='val_accuracy',         # Monitor validation loss (or accuracy)
    save_best_only=True,        # Only save the best model
    mode='max',                 # We want to minimize the validation loss
    verbose=1
)
validation_generator = train_generator
# Training the model with validation data and checkpoint
model.fit(
    train_generator, 
    epochs=10,
    validation_data=validation_generator,  # Add the validation data generator here
    callbacks=[checkpoint_callback]  # Add the checkpoint callback to save the best model
)

Epoch 1/10
24/63 ━━━━━━━━━━━━━━━━━━━━ 40:33 62s/step - accuracy: 0.8647 - loss: 1.0277